In [ ]:
%pip install langchain_ollama

In [ ]:
import json
import os

from langchain_ollama import ChatOllama

from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
    AIMessage
)

In [ ]:
# -----------------------------
# LOCAL MODEL
# -----------------------------
llm = ChatOllama(
    model="llama3"
)


In [ ]:
# -----------------------------
# SYSTEM PROMPT
# -----------------------------
import os

# Create the 'prompts' directory if it doesn't exist
if not os.path.exists('prompts'):
    os.makedirs('prompts')

# Example content for system_prompt.txt
system_prompt = (
    """You are a professional AI assistant.

      Rules:
        - Be concise
        - Answer clearly
        - Keep context from conversation"""
)

# Write the content to the file
with open('prompts/system_prompt.txt', 'w') as file:
    file.write(system_prompt)

print("Created 'prompts/system_prompt.txt' with example content.")


In [ ]:
# -----------------------------
# MEMORY FILE
# -----------------------------
MEMORY_FILE = "memory/chat_history.json"

In [ ]:
# -----------------------------
# INITIALIZE CHAT HISTORY
# -----------------------------
chat_history = [
    SystemMessage(content=system_prompt)
]

# Ensure the 'memory' directory exists
memory_dir = os.path.dirname(MEMORY_FILE)
if not os.path.exists(memory_dir):
    os.makedirs(memory_dir)

# Create memory file if missing
if not os.path.exists(MEMORY_FILE):

    with open(MEMORY_FILE, "w") as file:
        json.dump([], file)

# Load previous chats
with open(MEMORY_FILE, "r") as file:

    stored_messages = json.load(file)

for msg in stored_messages:

    if msg["role"] == "user":

        chat_history.append(
            HumanMessage(content=msg["content"])
        )

    elif msg["role"] == "assistant":

        chat_history.append(
            AIMessage(content=msg["content"])
        )


In [ ]:
# -----------------------------# CHAT LOOP# -----------------------------
print("\n🤖 Local AI Chatbot Started!")
print("Running with Llama 3 locally.")
print("Type 'exit' to quit.\n")
while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        print("\n👋 Goodbye!")
        break
    # Add user message
    user_message = HumanMessage(content=user_input)
    chat_history.append(user_message)
    # Generate AI response
    try:
        response = llm.invoke(chat_history)
        ai_reply = response.content
    except Exception as e:
        print(f"\n⚠️ Error communicating with Ollama: {e}\nPlease ensure Ollama server is running and 'llama3' model is available.\n")
        # Remove the last user message from history if the AI call failed
        chat_history.pop()
        continue
    # Print response
    print(f"\nAI: {ai_reply}\n")
    # Save AI message
    ai_message = AIMessage(content=ai_reply)
    chat_history.append(ai_message)


In [ ]:
# -----------------------------
# SETUP OLLAMA IN COLAB
# -----------------------------

import subprocess
import threading
import time
import requests
import os

def start_ollama():
    # Kill any existing Ollama processes
    subprocess.run(['killall', '-q', 'ollama'], check=False)

    # Start Ollama in the background
    # Use nohup to ensure it runs even if the parent process exits
    # and redirect output to a log file
    command = "ollama serve"
    process = subprocess.Popen(command.split(), stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    # Wait for Ollama to start (check for a specific log line or port binding)
    print("Waiting for Ollama to start...")
    time.sleep(5) # Give it a few seconds to boot up

    # Check if Ollama is responsive
    try:
        requests.get('http://localhost:11434')
        print("Ollama server is running.")
        return True
    except requests.exceptions.ConnectionError:
        print("Ollama server did not start correctly.")
        print("Stdout:", process.stdout.read().decode())
        print("Stderr:", process.stderr.read().decode())
        return False

def pull_model(model_name="llama3"):
    print(f"Pulling model: {model_name}")
    command = f"ollama pull {model_name}"
    # Run ollama pull with stderr redirected to stdout to capture progress
    result = subprocess.run(command.split(), capture_output=True, text=True)
    if result.returncode == 0:
        print(f"Model {model_name} pulled successfully.")
    else:
        print(f"Failed to pull model {model_name}.\nStderr: {result.stderr}\nStdout: {result.stdout}")
        # If pull fails, Ollama server might not be running or model name is wrong.
        # This might be an indication to re-check the server status.
        print("Attempting to re-check Ollama server status after model pull failure...")
        if not start_ollama(): # Try to restart/verify server if pull failed
             print("Ollama server is still not responding. Please check manual setup steps.")


# Main execution
print("Downloading Ollama...")

# Manual installation steps for Ollama in Colab
# Updated URL to a known working version of the Ollama binary
ollama_url = "https://github.com/ollama/ollama/releases/download/v0.1.29/ollama-linux-amd64"
ollama_bin_path = "/usr/local/bin/ollama"

try:
    # Download the Ollama binary
    wget_result = subprocess.run(['wget', ollama_url, '-O', ollama_bin_path], check=True, capture_output=True, text=True)
    print("Wget stdout:", wget_result.stdout)
    print("Wget stderr:", wget_result.stderr)

    # Make it executable
    subprocess.run(['chmod', '+x', ollama_bin_path], check=True)
    print("Ollama binary downloaded and made executable.")

except subprocess.CalledProcessError as e:
    print(f"Failed to download or setup Ollama binary: {e}")
    print(f"Stderr: {e.stderr}")
    print(f"Stdout: {e.stdout}")
    print("Ollama setup failed. Please check the output above for errors.")
else:
    # Only proceed if installation was successful
    if start_ollama():
        pull_model("llama3")
        print("Ollama setup complete. You can now run the chat loop again.")
    else:
        print("Ollama server did not start after binary setup. Please check the output above for errors.")


In [ ]:
# -----------------------------
# SAVE MEMORY
# -----------------------------
save_messages = []

for msg in chat_history:

    if isinstance(msg, SystemMessage):
        continue

    elif isinstance(msg, HumanMessage):

        save_messages.append({
            "role": "user",
            "content": msg.content
        })

    elif isinstance(msg, AIMessage):

        save_messages.append({
            "role": "assistant",
            "content": msg.content
        })

with open(MEMORY_FILE, "w") as file:
    json.dump(save_messages, file, indent=4)
